# RAG 기반 AI 수학 학습 튜터 PoC

기간: **2025.06.09 ~ 2025.06.27**

원본 Colab의 핵심 실험을 정리한 공개용 노트북입니다. API 키 입력, 데이터 경로, 설치 명령을 정리하고 대량 데이터 출력과 미사용 셀을 제거했습니다. 원본 모델명과 검색·프롬프트 설정은 유지했습니다. 저장된 출력은 제거했으며 원본 관찰 내용은 `docs/experiment-notes.md`에 기록했습니다.

원본의 모델명은 실험 당시 설정입니다. 재실행 시 이용 가능한 모델명과 패키지 호환성을 확인하세요. 원본 데이터는 별도로 준비해야 합니다.

# 0. 설치 및 준비

In [ ]:
import os
from getpass import getpass

# API 키를 코드에 기록하지 않습니다.
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Google API key: ")


In [ ]:
%pip install "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langchain-google-genai>=2,<3" chromadb


In [ ]:
# 원본 실험 모델명. 필요하면 환경변수로 변경합니다.
CHAT_MODEL = os.environ.get("CHAT_MODEL", "gemini-1.5-flash")
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "models/text-embedding-004")


In [ ]:
#랭체인의 구글Api를 사용합니다
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model=CHAT_MODEL, temperature=0.0)

# 1. JSON Loader

In [ ]:
import json
from pathlib import Path

# learning_data_info를 포함하는 JSON 배열 파일을 지정하세요.
file_path = Path(os.environ.get("MATH_DATA_PATH", "../data/m1_t_merged.json"))
data = json.loads(file_path.read_text(encoding="utf-8"))


# 2. 텍스트 분할

In [ ]:
import json
from langchain.docstore.document import Document

docs = []
for idx, item in enumerate(data):
    learning_info = item.get("learning_data_info", {})

    # 필요한 텍스트 필드만 선택 (null 방지)
    text_parts = [
        learning_info.get("text_an", ""),
        learning_info.get("text_description", ""),
        learning_info.get("text_qa", "")
    ]
    combined_text = "\n".join([part for part in text_parts if isinstance(part, str) and part.strip()])

    if combined_text:  # 실제 내용이 있다면
        doc = Document(
            page_content=combined_text,
            metadata={
                "id": learning_info.get("learning_data_name", f"doc_{idx}"),
                "class_name": learning_info.get("class_name", ""),
            }
        )
        docs.append(doc)

print(f"✅ Document 개수: {len(docs)}")

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,      # chunk 당 문자 수
    chunk_overlap=50,    # 중복 영역 (문맥 유지)
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(docs)  # docs는 Document 객체 리스트
print(f"✅ 분할된 청크 수: {len(chunks)}")

# 3. Embedding

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embedding_model = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

# 4. VectorStore

In [ ]:
from langchain.vectorstores import Chroma

In [ ]:
vectorstore = Chroma.from_documents(chunks, embedding=embedding_model, persist_directory="./chroma_index")

In [ ]:
query = "반지름이 8cm인 구의 부피를 구하는 방법을 알려줘"
search_docs = vectorstore.similarity_search(query)

print(search_docs[0].page_content)

# 5. Retriever

In [ ]:
from langchain.chains import RetrievalQA


In [ ]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=vectorstore.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

In [ ]:
query = "반지름이 8cm인 구의 부피를 구하는 방법을 알려줘"
result = qa.invoke({"query": query})

from IPython.display import Markdown, display
display(Markdown(result["result"]))

# 6. 추가 실험

In [ ]:
retrieved_docs = vectorstore.similarity_search(query, k=3)

In [ ]:
context = "\n\n---\n\n".join([doc.page_content for doc in retrieved_docs])
prompt = f"""
당신은 중학교 수학 교사입니다.
아래 문서들을 참고하여 '{query}'라는 질문에 대해 중학교 1학년 수준으로 설명해 주세요.

[문서]
{context}
"""


In [ ]:
from langchain.schema import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="당신은 친절한 중학교 수학 선생님입니다."),
    HumanMessage(content=prompt)
]

response = llm.invoke(messages)
print(response.content)

In [ ]:
llm2 = ChatGoogleGenerativeAI(model=CHAT_MODEL, temperature=0.0)
request = llm2.invoke("반지름이 8cm인 구의 부피를 구하는 방법을 알려줘")
display(Markdown(request.content))